In [ ]:
# RTX 5090 OCR 极速冲刺 80 分工作流
这个 Notebook 专门为 AutoDL 等云端 5090 环境设计，集成了环境安装、数据准备、模型训练和预测流程。

### 1. 环境基础配置与库安装
安装 PaddlePaddle 以及项目所需的依赖。针对 5090 建议使用最新的 CUDA 12.x 驱动。

In [ ]:
# 针对 AutoDL 5090 进行环境优化
# 1. 升级 Paddle 以适配 5090 的算子加速
%pip install paddlepaddle-gpu==3.0.0b1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

# 2. 补全 OCR 训练必备依赖
%pip install pandas pillow numpy tqdm

# 3. 验证是否安装成功并能识别 5090
import paddle
print("当前 Paddle 版本:", paddle.__version__)
print("GPU 是否可用:", paddle.device.is_compiled_with_cuda())
print("环境准备完成！")


### 2. 检查硬件加速环境
确认 5090 显卡已由系统识别。建议查看显卡利用率及显存状态。

In [ ]:
!nvidia-smi
import paddle
print("Paddle Version:", paddle.__version__)
print("GPU Available:", paddle.device.is_compiled_with_cuda())
print("Device Name:", paddle.get_device())

### 3. 数据集与项目代码解压
解压上传到 `/root/` 或当前目录下的数据集压缩包，整理目录结构。

In [ ]:
# 针对分卷压缩包和嵌套文件夹进行处理
import os
import shutil

# 1. 合并分卷并解压
if os.path.exists('data_archive.zip'):
    print("检测到分卷压缩包，正在合并并在当前目录解压...")
    # 合并分卷
    !zip -F data_archive.zip --out data_full.zip
    # 直接解压到当前目录，防止重复创建 data 文件夹
    !unzip -q data_full.zip -d ./
    print("解压完成！")
else:
    print("未发现 data_archive.zip")

# 2. 修复嵌套文件夹 (处理 data/data/train_images 这种情况)
if os.path.exists('data/data'):
    print("检测到嵌套文件夹，正在移动内容到一级 data 目录...")
    # 获取子 data 目录下的所有内容
    nested_dir = 'data/data'
    target_dir = 'data'
    # 先移动内容，再删掉空的子 data
    for item in os.listdir(nested_dir):
        s = os.path.join(nested_dir, item)
        d = os.path.join(target_dir, item)
        if os.path.exists(d):
            if os.path.isdir(d): shutil.rmtree(d)
            else: os.remove(d)
        shutil.move(s, d)
    os.rmdir(nested_dir)
    print("文件夹修复完成！")

# 3. 运行数据读取脚本
if os.path.exists('prepare_data.py'):
    print("正在运行 prepare_data.py 生成训练索引...")
    !python prepare_data.py
    print("数据准备完成！")


### 4. 顺序执行训练与推理脚本
调用主要的训练脚本 `train_scratch.py` 以及用于生成最终结果的 `predict.py`。
由于 BS=1024 效率极高，请关注显存占用。如果遇到 OOM 报错建议手动下调 BS 再运行。

In [ ]:
# 开始训练（狂暴模式）
!python train_scratch.py

# 训练完成后，生成 submission.csv 用于上传
!python predict.py
print("预测流程执行结束，请在项目根目录下寻找 result.txt 和 submission.csv 文件。")

### 5. 模型权重验证与结果展示
检查输出目录生成的权重文件，并预览预测结果以验证准确率。如有需要可手动在 local 继续微调。

In [ ]:
# 导出最终结果供本地下载
# !zip -r my_model_output.zip checkpoints result.txt submission.csv
!ls -lh checkpoints/best_model.pdparams
print("恭喜！流程已全部走通，快去 leaderboard 提交测试吧。")